In [ ]:
#| default_exp core

# aplnb core
> Driving Dyalog APL over the RIDE protocol, and `apl` magics for Jupyter and IPython


In [ ]:
import atexit,html,json,socket,subprocess
from shutil import which
from importlib.resources import files
from fastcore.utils import *
from fastcore.test import *
from IPython.display import display, Javascript, HTML
from IPython.paths import get_ipython_dir


aplnb runs Dyalog APL from Python and provides `apl` magics for Jupyter and IPython. It uses the [RIDE protocol](https://github.com/Dyalog/ride/blob/master/docs/protocol.md), as do Dyalog's IDE and official [Jupyter kernel](https://github.com/Dyalog/dyalog-jupyter-kernel). It needs no extra code loaded into the APL workspace.

RIDE messages distinguish output, errors, and readiness for input. The examples below show the protocol before building the `Apl` session object.

## Finding and starting Dyalog

In [ ]:
def find_dyalog():
    "Locate the Dyalog interpreter binary"
    if p:=which('mapl') or which('dyalog'): return p
    apps = sorted(Path('/Applications').glob('Dyalog-*.app'))
    if apps: return str(apps[-1]/'Contents/Resources/Dyalog/mapl')
    vers = sorted(Path('/opt/mdyalog').glob('*/*/*/mapl'))
    if vers: return str(vers[-1])
    raise FileNotFoundError('Dyalog APL not found: install it from dyalog.com')

`find_dyalog` first looks for `mapl` or `dyalog` on `PATH`. It then checks standard installation directories on macOS and Linux.

In [ ]:
find_dyalog()

'/usr/local/bin/dyalog'

RIDE supports two connection directions. With `RIDE_INIT=SERVE:*:port`, the interpreter listens for a client. Dyalog's Jupyter kernel uses this mode and polls for the listening port.

aplnb uses `CONNECT` mode. It listens on a port allocated by the OS, then starts Dyalog with that address. `accept()` waits for the interpreter to connect. This avoids selecting an unused port before binding it. The environment also sets `RIDE_SPAWNED=1`; explicit shutdown is covered below.

In [ ]:
def start_dyalog(
    dyalog=None,  # Path to the interpreter binary; `find_dyalog()` result if None
    timeout=10,   # Socket timeout (secs), so a client bug can never hang the caller
):
    "Spawn a Dyalog interpreter that connects back to us over RIDE; return `(socket,Popen)`"
    if not dyalog: dyalog = find_dyalog()
    lsn = socket.create_server(('127.0.0.1', 0))
    port = lsn.getsockname()[1]
    env = os.environ | dict(RIDE_INIT=f'CONNECT:127.0.0.1:{port}', RIDE_SPAWNED='1', MAXAPLCORES=os.environ.get('MAXAPLCORES','0'),
        DYALOGQUIETUCMDBUILD='1', DYALOG_LINEEDITOR_MODE='1', ENABLE_CEF='0', LOG_FILE_INUSE='0')
    dn = subprocess.DEVNULL
    proc = subprocess.Popen([dyalog], env=env, stdin=dn, stdout=dn, stderr=dn)
    lsn.settimeout(10)
    sock,_ = lsn.accept()
    lsn.close()
    sock.settimeout(timeout)
    return sock,proc

In [ ]:
sock,proc = start_dyalog()


## Message framing

The interpreter speaks first. Here are the raw bytes:

In [ ]:
raw = sock.recv(64)
raw

b'\x00\x00\x00\x1cRIDESupportedProtocols=2'

A RIDE message contains a 4-byte big-endian total length, the literal `RIDE`, and a UTF-8 payload. Here `0x1c` is 28: 4 length bytes, 4 bytes for `RIDE`, and 20 for `SupportedProtocols=2`.

The first two messages in each direction negotiate the protocol version as plain strings. Later payloads are JSON arrays containing a command name and its arguments.

Dyalog sometimes sends raw control characters inside JSON strings. `ride_recv` escapes these before decoding, as does the official Jupyter kernel.

In [ ]:
def ride_send(sock, msg):
    "Send one RIDE message: a handshake `str`, or a `[cmd,args]` list sent as JSON"
    if not isinstance(msg,str): msg = json.dumps(msg, separators=(',',':'))
    b = ('RIDE'+msg).encode()
    sock.sendall((len(b)+4).to_bytes(4,'big')+b)

def _recvall(sock, n):
    parts = []
    while n:
        b = sock.recv(n)
        if not b: raise ConnectionError('Dyalog closed the connection')
        parts.append(b)
        n -= len(b)
    return b''.join(parts)

def ride_recv(sock):
    "Receive one RIDE message, JSON-decoded unless it's a handshake string"
    hdr = _recvall(sock, 8)
    assert hdr[4:8]==b'RIDE', f"Bad RIDE header: {hdr}"
    msg = _recvall(sock, int.from_bytes(hdr[:4],'big')-8).decode()
    msg = re.sub(r'[\x00-\x1f]', lambda m: f'\\u{ord(m[0]):04x}', msg)
    return json.loads(msg) if msg[0]=='[' else msg


Each side sends `SupportedProtocols=2` and `UsingProtocol=2`. The client then sends `Identify` and reads the interpreter's details:

In [ ]:
ride_send(sock, 'SupportedProtocols=2')
ride_send(sock, 'UsingProtocol=2')
ride_recv(sock)

'UsingProtocol=2'

In [ ]:
ride_send(sock, ['Identify',{'apiVersion':1,'identity':1}])
info = ride_recv(sock)[1]
{k:info[k] for k in ('Vendor','Language','version','arch','platform')}


{'Vendor': 'Dyalog Limited',
 'Language': 'APL',
 'version': '20.0.53963',
 'arch': 'Unicode/64',
 'platform': 'Mac-64'}

Wait for `SetPromptType` with `type` 1, which signals readiness for input:

In [ ]:
msgs = []
while True:
    m = ride_recv(sock)
    msgs.append(m)
    if m[0]=='SetPromptType' and m[1]['type']==1: break
msgs

[['UpdateDisplayName', {'displayName': 'CLEAR WS'}],
 ['SetPromptType', {'type': 1}]]

## Executing code

`Execute` sends session input with a required trailing newline. Dyalog echoes the input and sends output in `AppendSessionOutput` messages. `SetPromptType` 1 signals that execution has finished and the interpreter is ready for more input.

In [ ]:
ride_send(sock, ['Execute',{'text':'3×⍳4\n','trace':0}])
msgs = []
while True:
    m = ride_recv(sock)
    msgs.append(m)
    if m[0]=='SetPromptType' and m[1]['type']==1: break
msgs

[['UpdateSessionCaption', {'text': 'CLEAR WS - Dyalog APL'}],
 ['AppendSessionOutput', {'result': '3×⍳4\n', 'type': 14, 'group': 0}],
 ['SetPromptType', {'type': 0}],
 ['AppendSessionOutput', {'result': '3 6 9 12\n', 'type': 2, 'group': 0}],
 ['SetPromptType', {'type': 1}]]

Output type 14 is an input echo. `HadError` reports an error independently of its displayed text:

In [ ]:
ride_send(sock, ['Execute',{'text':'1÷0\n','trace':0}])
msgs = []
while True:
    m = ride_recv(sock)
    msgs.append(m)
    if m[0]=='SetPromptType' and m[1]['type']==1: break
msgs

[['AppendSessionOutput', {'result': '1÷0\n', 'type': 14, 'group': 0}],
 ['SetPromptType', {'type': 0}],
 ['HadError', {'error': 11, 'dmx': 1}],
 ['AppendSessionOutput',
  {'result': 'DOMAIN ERROR: Divide by zero\n', 'type': 5, 'group': 0}],
 ['AppendSessionOutput', {'result': '      1÷0\n', 'type': 5, 'group': 0}],
 ['AppendSessionOutput', {'result': '       ∧\n', 'type': 5, 'group': 0}],
 ['SetPromptType', {'type': 1}]]

`AplError` includes Dyalog's error display. Its `reset` flag indicates that the interpreter was replaced and workspace state was lost. `AplPrompt` is an internal exception for prompts other than the ready prompt.

In [ ]:
class AplError(Exception):
    "An APL error, carrying the session's error display as its message; `reset` means the interpreter was replaced and workspace state lost"
    def __init__(self, msg, reset=False):
        super().__init__(msg)
        self.reset = reset

class AplPrompt(Exception):
    "The session stopped at a non-ready prompt: 2=⎕ input, 3=incomplete input, 4=⍞ input"
    def __init__(self, ptype):
        super().__init__(f'prompt type {ptype}')
        self.ptype = ptype


`ride_run` sends all non-blank input lines in one `Execute` message. It collects output, excluding input echoes and prompt strings. The result is `(output, errno)`, with error number 0 for success.

A final prompt counts only after Dyalog has echoed every input line. This matters for multiline blocks, as the next example shows.


In [ ]:
def ride_run(sock, code):
    "Run one or more lines of APL in the session; return `(output,errno)` once the session is ready again"
    lines = [l for l in code.splitlines() if l.strip()]
    ride_send(sock, ['Execute',{'text':'\n'.join(lines)+'\n','trace':0}])
    out,err,pending = '',0,len(lines)
    while True:
        m,a = ride_recv(sock)
        if m=='AppendSessionOutput':
            if a['type']==14: pending -= 1
            elif a['type']!=1: out += a['result']
        elif m=='HadError': err = a['error']
        elif m=='SetPromptType' and a['type']!=0 and not pending:
            if a['type']!=1: raise AplPrompt(a['type'])
            return out,err


## Multi-line input

Dyalog 20.0 enables multiline session input by default. Send a complete block in one `Execute` message. The interpreter echoes each line and uses prompt type 3 while the block is open.

A type 3 prompt before all line echoes means Dyalog is still reading the block. After all echoes, it means the submitted block is incomplete. Do not send another `Execute` at that prompt: the interpreter can crash. `Apl.run` handles incomplete input by replacing the interpreter.

In [ ]:
ride_run(sock, '''
:If 1
    ⎕←6×7
:EndIf''')


('42\n', 0)

Session state persists across calls:


In [ ]:
test_eq(ride_run(sock, 'x←⍳3\nz←x×x\n⎕←z'), ('1 4 9\n', 0))


Errors come back with the `HadError` number and the session error display:

In [ ]:
out,err = ride_run(sock, '1÷0')
print(out)
test_eq(err, 11)


DOMAIN ERROR: Divide by zero
      1÷0
       ∧



## A session object

`Apl` starts Dyalog and completes the handshake before accepting input. It sets the print width to 32767, matching the official kernel, to avoid wrapping long output.

In [ ]:
class Apl:
    "A Dyalog APL session over the RIDE protocol"
    def __init__(self, dyalog=None, timeout=10):
        store_attr()
        self._connect()

    def _connect(self):
        self.sock,self.proc = start_dyalog(self.dyalog, self.timeout)
        assert ride_recv(self.sock)=='SupportedProtocols=2'
        ride_send(self.sock, 'SupportedProtocols=2')
        ride_send(self.sock, 'UsingProtocol=2')
        assert ride_recv(self.sock)=='UsingProtocol=2'
        ride_send(self.sock, ['Identify',{'apiVersion':1,'identity':1}])
        self.info = ride_recv(self.sock)[1]
        ride_send(self.sock, ['SetPW',{'pw':32767}])
        atexit.register(self.close)
        while True:
            m,a = ride_recv(self.sock)
            if m=='SetPromptType' and a['type']==1: break
        self.sock.settimeout(None)  # timeout guards startup only; run() waits as long as the code takes


`close` sends `Exit` and waits up to three seconds. It kills the process if it has not exited. `_connect` registers this cleanup with `atexit` to avoid leaving Dyalog processes behind. Closing the socket alone does not reliably shut down a stuck interpreter.

In [ ]:
@patch
def close(self:Apl):
    "Shut down the interpreter and close the connection"
    atexit.unregister(self.close)
    try: ride_send(self.sock, ['Exit',{'code':0}])
    except OSError: pass
    self.sock.close()
    try: self.proc.wait(3)
    except subprocess.TimeoutExpired:
        self.proc.kill()
        self.proc.wait()

In [ ]:
apl = Apl()
apl.info['version']

'20.0.53963'

`run` returns output or raises `AplError`:

- Ordinary APL errors include the session's error display.
- Requests for `⎕` or `⍞` input are cancelled. The session remains usable.
- Incomplete input, such as an unclosed `:If`, requires a new interpreter because of [Dyalog/ride#1401](https://github.com/Dyalog/ride/issues/1401). The exception has `reset=True`, indicating that workspace state was lost.

In [ ]:
@patch
def run(self:Apl, code):
    "Run `code` in the session, returning its output; raises `AplError` on APL errors"
    try: out,err = ride_run(self.sock, code)
    except AplPrompt as e:
        if e.ptype in (2,4):
            ride_run(self.sock, '→' if e.ptype==2 else '')
            raise AplError('Input via ⎕ or ⍞ is not supported in aplnb') from None
        self.close()
        self._connect()
        raise AplError('Incomplete input wedged the Dyalog interpreter; started a fresh session (workspace lost). See Dyalog/ride#1401',
            reset=True) from None
    if err: raise AplError(out)
    return out


In [ ]:
print(apl.run('m←2 3⍴⍳6\n⎕←m'))

1 2 3
4 5 6



The matrix remains available for later calls. Ordinary APL errors do not reset the session:

In [ ]:
test_eq(apl.run('+/,m'), '21\n')
test_fail(lambda: apl.run('m+\'x\''), contains='DOMAIN ERROR')

Calling `apl(code)` displays output without an extra `print`. It returns `AplOut`, a `str` subclass that preserves Dyalog's formatting in notebook displays. Results still support ordinary string operations. Calls with no output return `None`.

In [ ]:
class AplOut(str):
    "Output text from an `Apl` call; displays verbatim, in the SAX2 APL font where HTML is available"
    def __repr__(self): return str(self)
    def _repr_html_(self): return f'<pre class="aplnb_out sax2">{html.escape(self.rstrip(chr(10)))}</pre>'

@patch
def __call__(self:Apl, code):
    "Run `code`, returning session output (or None if there is none)"
    return AplOut(self.run(code)) or None


In [ ]:
apl('m ∘.× ⍳4')

1  2  3  4
2  4  6  8
3  6  9 12
          
4  8 12 16
5 10 15 20
6 12 18 24

In [ ]:
test_eq(apl('+/,m'), '21\n')
test_is(apl('m2←m×10'), None)

## Getting values into Python

`pyval` returns a Python value instead of display text. It asks Dyalog to serialize the expression with `⎕JSON`, then parses the JSON.

The `HighRank` option serializes arrays of rank 2 or higher as nested lists. Without it, `⎕JSON` rejects these arrays. The left argument `1` forces serialization. Monadic `⎕JSON` would try to parse a character vector as JSON.

In [ ]:
@patch
def pyval(self:Apl, expr):
    "Evaluate `expr` and return the result as a Python value"
    return json.loads(self.run(f"1(⎕JSON⍠'HighRank' 'Split')({expr})"))

In [ ]:
test_eq(apl.pyval('m'), [[1,2,3],[4,5,6]])
test_eq(apl.pyval('3×⍳4'), [3,6,9,12])
test_eq(apl.pyval('⎕A'), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ')

Use square brackets to read an APL expression as a Python value or assign a Python value to an APL variable:

In [ ]:
def _apljson(v):
    "An APL expression that evaluates to the Python value `v`"
    return "⎕JSON'" + json.dumps(v).replace("'", "''") + "'"

@patch
def __getitem__(self:Apl, expr): return self.pyval(expr)

@patch
def __setitem__(self:Apl, nm, v): self.run(f'{nm}←{_apljson(v)}')

In [ ]:
apl['q'] = [[1,2],[3,4.5]]
test_eq(apl['q'], [[1,2],[3,4.5]])
test_eq(apl["'nested: ',⍕⎕NC'q'"], 'nested: 2')
apl['s'] = "it's"
test_eq(apl['s'], "it's")

`⎕JSON` imports nested lists as vectors of vectors. Use `↑` to make a rank-2 matrix:

In [ ]:
apl('↑q')

1 2  
3 4.5

`fn` returns a Python callable for an APL function. One argument calls it monadically. Two arguments call it dyadically, with the left argument first. Arguments and results use the same JSON conversions as `pyval` and assignment.

In [ ]:
@patch
def fn(self:Apl, code):
    "A Python callable applying APL function `code` monadically or dyadically"
    def f(*args):
        if len(args)==1: return self.pyval(f'({code}){_apljson(args[0])}')
        a,w = args
        return self.pyval(f'({_apljson(a)})({code}){_apljson(w)}')
    return f

In [ ]:
sq = apl.fn('{⍵*2}')
test_eq(sq([1,2,3]), [1,4,9])
test_eq(apl.fn('+/')([1,2,3]), 6)
test_eq(apl.fn('↑')(2, [5,6,7]), [5,6])

Use a context manager to close a session before process exit:

In [ ]:
@patch
def __enter__(self:Apl): return self

@patch
def __exit__(self:Apl, *args): self.close()

In [ ]:
with Apl() as a2: test_eq(a2('2+2'), '4\n')

## Input requests and incomplete input

Check that cancelled input requests leave the session usable:

In [ ]:
test_fail(lambda: apl('x←⎕'), contains='not supported')
test_eq(apl('2+2'), '4\n')

In [ ]:
test_fail(lambda: apl('x←⍞'), contains='not supported')
test_eq(apl('2+2'), '4\n')

Check that incomplete input resets the workspace:

In [ ]:
try: apl(':If 1')
except AplError as ex: err = ex
test_eq(err.reset, True)
assert 'wedged' in str(err)
test_eq(apl('2+2'), '4\n')


The `timeout` parameter limits startup. Once connected, `Apl.run` waits as long as the computation takes:

In [ ]:
with Apl(timeout=2) as a2: assert float(a2.run('⎕←⎕DL 3')) >= 3


## The `apl` magics

`%%apl` runs a cell and displays its session output. A trailing `;` suppresses the display. `%apl expr` returns a Python value, as with `apl[expr]`, and can appear in an assignment: `z = %apl z`.

The first magic call starts the interpreter and adds the [APL language bar](https://abrudz.github.io/lb/apl) to the page. Registration alone does not start Dyalog.

Output uses Adám Brudzewsky's [SAX2](https://github.com/abrudz/SAX2) APL font, loaded locally or from a CDN. Monospace is the fallback when SAX2 is unavailable.

In [ ]:
_css = """<style>
@font-face { font-family:'SAX2'; src: local('SAX2'), url('https://cdn.jsdelivr.net/gh/abrudz/SAX2@master/SAX2.ttf') format('truetype') }
.sax2 { font-family:'SAX2',monospace !important; line-height:1.05 !important }
</style>"""

class APLMagic:
    "IPython `%apl`/`%%apl` magics, driving a lazily-started `Apl` session"
    def __init__(self, dyalog=None): self.dyalog,self.o,self._loaded = dyalog,None,False

    def apl(self, line, cell=None):
        "Run APL: a cell magic displays the session output; a line magic returns the expression's Python value"
        if not self.o: self.o = Apl(self.dyalog)
        if not self._loaded:
            display(Javascript((files('aplnb')/'lb.js').read_text()))
            display(HTML(_css))
            self._loaded = True
        if cell is None: return self.o[line.split('⍝')[0].strip()]
        disp,cell = True,cell.rstrip()
        if cell.endswith(';'): disp,cell = False,cell[:-1]
        out = self.o(cell)
        if disp and out: display(out)


In [ ]:
def create_magic(shell=None):
    "Create an `APLMagic` and register its `apl` line/cell magic with `shell`, returning it"
    if not shell: shell = get_ipython()
    apl_magic = APLMagic()
    shell.register_magic_function(apl_magic.apl, 'line_cell', 'apl')
    return apl_magic


In [ ]:
# Only required if you don't load the extension
magic = create_magic()


In [ ]:
%%apl
m2←3 3⍴⍳9
⎕←m2

Javascript(// APL language bar by Adám Brudzewsky: https://abrudz.github.io/lb (source: https://github.com/abrudz/lb)
// MIT License, Copyright (c) 2011-2020 Nikolay G. Nikolov and Adam Brudzevski. This is a modified copy bundled with iversonnb.
// Changes from upstream: double backtick composes ```; insertion via insertText so undo and input events work;
// Monaco editor support (incl. EditContext mode); dark mode; overlay/push-down toggle persisted per site;
// idempotent injection; ResizeObserver-driven layout; @font-face with dead url() removed; skipped on quarto-rendered pages.
; (_ => {
	if (document.querySelector('.ngn_lb')) return
	if (document.querySelector('meta[name=generator][content^=quarto]')) return //no bar on rendered docs pages
	let hc = { '<': '&lt;', '&': '&amp;', "'": '&apos;', '"': '&quot;' }, he = x => x.replace(/[<&'"]/g, c => hc[c]) //html chars and escape fn
		, tcs = '<-←xx×/\\×:-÷*O⍟[-⌹-]⌹OO○77⌈FF⌈ll⌊LL⌊T_⌶II⌶|_⊥TT⊤-|⊣|-⊢=/≠L-≠<=≤<_≤>=≥>_≥==≡=_≡7=≢Z-≢vv∨^^∧^

HTML(<style>
@font-face { font-family:'SAX2'; src: local('SAX2'), url('https://cdn.jsdelivr.net/gh/abrudz/SAX2@master/SAX2.ttf') format('truetype') }
.sax2 { font-family:'SAX2',monospace !important; line-height:1.05 !important }
</style>)

1 2 3
4 5 6
7 8 9

The line magic brings values back into Python:

In [ ]:
z = %apl m2  ⍝ comments are fine here too
test_eq(z, [[1,2,3],[4,5,6],[7,8,9]])

The first example runs `]display`, a Dyalog user command. The second suppresses cell output with a trailing `;`.


In [ ]:
%%apl
]display 2 2⍴'ab' 'cd' 1 2

┌→──────────┐
↓ ┌→─┐ ┌→─┐ │
│ │ab│ │cd│ │
│ └──┘ └──┘ │
│           │
│ 1    2    │
│           │
└∊──────────┘

In [ ]:
%%apl
big←1000 1000⍴⍳12;

In [ ]:
#| hide
from IPython.utils.capture import capture_output

In [ ]:
#| hide
with capture_output() as cap: magic.apl('', '⍳3;')
test_eq(len(cap.outputs), 0)
with capture_output() as cap: magic.apl('', '⍳3')
test_eq(len(cap.outputs), 1)

In [ ]:
def load_ipython_extension(ipython):
    "Required function for creating magic"
    create_magic(shell=ipython)

In [ ]:
def create_ipython_config():
    "Called by `aplnb_install` to install magic"
    ipython_dir = Path(get_ipython_dir())
    cf = ipython_dir/'profile_default'/'ipython_config.py'
    cf.parent.mkdir(parents=True, exist_ok=True)
    if cf.exists() and 'aplnb' in cf.read_text(): return print('aplnb already installed!')
    with cf.open(mode='a') as f: f.write("\nc.InteractiveShellApp.extensions.append('aplnb')\n\n")
    print(f"Jupyter config updated at {cf}")

## Cleanup

Shut down the sessions this notebook started: the magic's, the `Apl` object's, and the raw-socket walkthrough one.

In [ ]:
magic.o.close()
apl.close()
ride_send(sock, ['Exit',{'code':0}])
sock.close()

## Export -

In [ ]:
#|hide
#|eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()